# Question 3: Demystifying Transformers – Building a Tiny Language Model

## Part A

### 3A.1.a

If $q$ and $k$ are random vectors whose entries are drawn from a distribution with mean 0 and variance 1, the variance of their dot product is going to be equal to the dimension and the mean is going to be 0. We know $q.k = \sum_i^{d_k} q_i . k_i$. Hence, 


$\mathbb{E}[q_i.k_i] = \mathbb{E}[q_i]\,\mathbb{E}[k_i] = 0$

$Var(q_i.k_i) = \mathbb{E}[q_i^2.k_i^2] - 0^2 = \mathbb{E}[q_i^2]\,\mathbb{E}[k_i^2] = 1$

Hence, 

$\mathbb{E}[q.k] = 0$
$Var[q.k] = \sum_i^{d_k} Var(q_i.k_i) = d_k$

If a query and key pair have large variance product, it would mean their values can reach very high numbers. The softmax over high attention scores is usually very unevenly distributed, with majority of the probability assigned to one or two tokens. When the values in softmax is very high, the gradients nears zero and no learning occurs in the network. Due to avoid getting such high probabilities, the scores are divided by a the term $\sqrt d_k$. On division the variance becomes one instead of $d_k$. 


### 3A.1.b

If we ommitted the scaling term, the gradient of softmax would tend to near zero, due to the reasons discussed above. 

In [ ]:
# for 3.A.2
import torch

matrix = torch.ones(4, 4)
ut = torch.triu(matrix, diagonal=1)
torch.masked_fill(ut, ut == 1, float('-inf'))

tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])

### 3A.2.a

tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])


### 3A.2.b

This is important in left to right language models as the main task for these models is to predict the next word using the context provided by the previous words. The main reason is how autoregressive model, an specifically autoregressive language models are formulated. They are formulated as $p(y_1, y_2, \dots, y_t) = \prod_t^T p(y_t| y_{<t})$. Firstly, this suggests that the language generation is modelled with the key constraint of the generation at a time step only depending upon the previous timesteps and not the later time steps. Secondly, it is also enforced because of parallel training and teacher forcing. If we did not do parallel training, rather trained the model sequentially at each step we would only compuyte the attention masks of the time step $t$ and prior time steps as the future time steps would not be in training. However due to parallel training the future tokens are going to be input and are goijng ot be input as ground truth due to teacher forcing. Due to this we need to avoid the future ground truth token and need causal masks to avoid them in the attenion score computation. 

We do not need them in BERT style models as the pretraining objective for them is bidirectional and are used for specific tasks. They were built as a single language representation for varied NLP tasks and used the whole input as a context. Due to this, at a particular time step the tokens from both the previous and future time steps are going to be used and we do not causal masking. 

### 3A.3.a

$PE(pos, 2i) = sin(pos/10000^{2i/d_model})$
$PE(pos, 2i+1) = cos(pos/10000^{2i/d_model})$

### 3A.3.b

Advantage: Sinusodal embeddings are hardcoded positional encodings, which have reasonable inductive bias of positions. However, at the scale of transformer training and according to scaling laws such hardcoded representations or alterations to the representations are better learned through large scale training. 

Disadvantage: In cases where training data for low resource languages are limited, positional encodings can provide meaningful position knowledge for autoregressive language modelling. Due to the limited scale the positional embeddings may not learn meaningful embeddings. Also learned embeddings cannot generalise to sequence length not seen in training. 

### 3A.4.a

The issue with Batch Norm when sequences in batch are padded to different true lengths is that the batch statistics in each iteration depend on the number of padding tokens, which may change for different lengths. For a small true length, the batch statistic can be completely distorted by padding tokens and the normalisation will be heavily batch dependent which we ironically want to avoid. Padded tokens are fillers to ensure data loading occurs with tensors of equal size, and they are masked during training. They also do not hold any semantic meaning, so they would not add anything meaningful to the batch statistic. Hence we require LayerNorm that normalises each token across its feature dimensions, rather than across the batch. This way the token is independent of any other token and hence there would not be any issues with the sequence length.

### 3A.4.b





### 3A.5.a

$Q \in R^{64\times64}$, $K \in R^{64\times64}$, $V \in R^{64\times64}$

### 3A.5.b

Multi head attention is more expressive than single head attention as it provides the model more capacity to represent concepts. What single head does is provides a single ranking/attention score amongst the tokens, representing which token is most important for the concept the model is learning at that layer/part of the model. However with multiple heads, we can represent multiple concepts on the account of each head calculating the normalised score(softmax score after calculating similarity scores amongst the tokens in the QKV projection space). 

In [2]:
import torch
from torch import nn
import torch.nn.functional as F


class layer(nn.Module):
    def __init__(self, emb_dim, heads, hidden_dim, vocab_size, norm_pos='pre', last_layer=False, context_len=512):
        super(layer, self).__init__()

        self.emb_dim = emb_dim
        self.heads = heads
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size

        self.q_proj = nn.Linear(emb_dim, emb_dim)
        self.k_proj = nn.Linear(emb_dim, emb_dim)
        self.v_proj = nn.Linear(emb_dim, emb_dim)
        self.proj_back = nn.Linear(emb_dim, emb_dim)

        self.ffn1 = nn.Linear(emb_dim, hidden_dim)
        self.ffn2 = nn.Linear(hidden_dim, emb_dim)

        self.attn_dropout = nn.Dropout(0.1)
        self.resid_dropout = nn.Dropout(0.1)
        self.ffn_dropout = nn.Dropout(0.1)

        self.norm_pos = norm_pos
        self.norm1 = nn.LayerNorm(emb_dim)
        self.norm2 = nn.LayerNorm(emb_dim)

        self.last_layer = last_layer
        if last_layer:
            self.last = nn.Linear(emb_dim, vocab_size)
    
    def forward(self, x):
        batch, seq_len, _ = x.shape
        inp = x.clone()
        if self.norm_pos == 'pre':
            x = self.norm1(x)

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        head_dim = self.emb_dim // self.heads
        q = q.view(batch, seq_len, self.heads, head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.heads, head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.heads, head_dim).transpose(1, 2)

        q_k_t = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
        masks = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(x.device)
        q_k_t = q_k_t.masked_fill(masks, float('-inf'))
        attn = torch.softmax(q_k_t, dim=-1)
        attn = self.attn_dropout(attn)
        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).reshape(batch, seq_len, self.emb_dim)

        out = self.proj_back(out)
        out = self.resid_dropout(out)
        out = out + inp

        if self.norm_pos == 'post':
            out = self.norm1(out)

        inp = out.clone()
        if self.norm_pos == 'pre':
            out = self.norm2(out)

        ffn_out = F.gelu(self.ffn1(out))
        ffn_out = self.ffn2(ffn_out)
        ffn_out = self.ffn_dropout(ffn_out)
        out = inp + ffn_out
        if self.norm_pos == 'post':
            out = self.norm2(out)

        if self.last_layer:
            out = self.last(out)
            return out

        return out

class LM(nn.Module):
    def __init__(self, emb_dim, heads, num_layers, hidden_dim, vocab_size, idx_to_char, char_to_idx, norm_pos='pre', context_len=128):
        super(LM, self).__init__()

        self.emb_dim = emb_dim
        self.heads = heads
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size

        self.char_emb = nn.Embedding(vocab_size, emb_dim)
        self.emb_dropout = nn.Dropout(0.1)

        self.layers = nn.ModuleList([layer(emb_dim, heads, hidden_dim, vocab_size, norm_pos) for _ in range(num_layers-1)])
        self.layers.append(layer(emb_dim, heads, hidden_dim, vocab_size, norm_pos, last_layer=True))

        self.idx_to_char = idx_to_char

        self.pos_emb = nn.Embedding(context_len, emb_dim)


    def forward(self, x):
        # x: b, seq_len, vocab_size
        out = self.char_emb(x)
        out += self.pos_emb(torch.arange(x.shape[1], device=x.device)).unsqueeze(0)
        out = self.emb_dropout(out)

        for layer in self.layers:
            out = layer(out)

        return out


    def generate(self, prompt, max_len=200, temperature=1.0):
        self.eval()
        output = ''
        with torch.no_grad():
            for i in range(max_len):
                out = self.forward(prompt)
                probs = F.softmax(out[:, -1] / temperature, -1)
                out = torch.argmax(probs, -1)
                char = self.idx_to_char[out.item()]
                output += char
                prompt = torch.cat([prompt, out.unsqueeze(0)], dim=1)

        return output


sample = torch.randint(0, 128, (4, 4)).long()
ascii_chars = [chr(i) for i in range(128)]

idx_to_char = {i: ch for i, ch in enumerate(ascii_chars)}
char_to_idx = {ch: i for i, ch in enumerate(ascii_chars)}

model = LM(emb_dim=32, heads=4, num_layers=2, hidden_dim=32, vocab_size=128, idx_to_char=idx_to_char, char_to_idx=char_to_idx)

out = model(sample)
torch.argmax(out, -1)

tensor([[ 22, 122,  25,  42],
        [ 16,  91,  49,  58],
        [ 30, 111,  25,  36],
        [ 30, 116,  25,  58]])

In [3]:
model.generate(sample[0:1])

IndexError: index out of range in self

Let say we now want to train a character level language model. What is the input the input is basically a sequence of text. We send it all together, and train it on the tokens at each time steps. First we need a normal english speaking dataset with paragraphs of text. then tokenise the characters. each unique character will get a token of their own, we will create a token map. he we will sample random sentences from the text and use them as training samples. the objective woul dbe to preidct the next word. 

In [ ]:
with open('corpus.txt', 'rb') as f:
    text = f.read()

In [ ]:
text_str = text.decode('ascii')

In [ ]:
unq = sorted(list(set(text_str)))
vocab_size = len(unq)
char_to_idx = {ch: i for i, ch in enumerate(unq)}
idx_to_char = {i: ch for i, ch in enumerate(unq)}

In [ ]:
num_samples = 1000000
poses = torch.randint(0, len(text_str)-130, (num_samples,))

train_x = []
train_y = []
for pos in poses:
    seq = text_str[pos:pos+128]
    target = text_str[pos+1:pos+129]

    train_x.append([char_to_idx[ch] for ch in seq])
    train_y.append([char_to_idx[ch] for ch in target])

In [ ]:
model = model.to("cuda")

In [ ]:
train_x = torch.tensor(train_x).to("cuda")
train_y = torch.tensor(train_y).to("cuda")

In [ ]:
model = LM(emb_dim=64, heads=4, num_layers=3, hidden_dim=256, vocab_size=vocab_size,idx_to_char=idx_to_char, char_to_idx=char_to_idx).to("cuda")

In [ ]:
def generate(model, prompt, max_len=200, temperature=1.0):
    model.eval()
    output = ''
    prompt = [char_to_idx[ch] for ch in prompt]
    prompt = torch.tensor(prompt).unsqueeze(0).to("cuda")
    with torch.no_grad():
        for i in range(max_len):
            out = model(prompt)
            probs = F.softmax(out[:, -1]/temperature, -1)
            out = torch.argmax(probs, -1)
            char = model.idx_to_char[out.item()]
            output += char
            prompt = torch.cat([prompt, out.unsqueeze(0)], dim=1)

    return output

model.load_state_dict(torch.load("ckpt/model_epoch30.pth"))
generate(model, prompt='The hero', max_len=200)

/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [128,0,0], thread: [0,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [128,0,0], thread: [1,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [128,0,0], thread: [2,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [128,0,0], thread: [3,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [128,0,0], thread: [4,0,0] Asser

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import os
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

train_x = train_x.long()
train_y = train_y.long()

# =========================
# train-test split
# =========================
split_ratio = 0.8
num_samples = len(train_x)

perm = torch.randperm(num_samples)
split_idx = int(split_ratio * num_samples)

train_idx = perm[:split_idx]
test_idx = perm[split_idx:]

x_train = train_x[train_idx]
y_train = train_y[train_idx]

x_test = train_x[test_idx]
y_test = train_y[test_idx]

batch_size = 4096
epochs = 1000

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# =========================
# learning rate scheduler
# =========================
# Option 1: StepLR
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=30,   # reduce LR every 50 epochs
    gamma=0.5       # new_lr = old_lr * 0.5
)

# Option 2: Uncomment this instead if you want LR to reduce
# when test loss stops improving.
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer,
#     mode='min',
#     factor=0.5,
#     patience=10
# )

os.makedirs("ckpt", exist_ok=True)

for epoch in range(epochs):
    # =========================
    # training
    # =========================
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    for start in range(0, len(x_train), batch_size):
        x_batch = x_train[start:start+batch_size].to(device)
        y_batch = y_train[start:start+batch_size].to(device)

        optimizer.zero_grad()
        out = model(x_batch)
        loss = loss_fn(out.reshape(-1, vocab_size), y_batch.reshape(-1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    train_loss = epoch_loss / num_batches

    # =========================
    # testing
    # =========================
    model.eval()
    test_loss = 0.0
    test_batches = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for start in range(0, len(x_test), batch_size):
            x_batch = x_test[start:start+batch_size].to(device)
            y_batch = y_test[start:start+batch_size].to(device)

            out = model(x_batch)
            loss = loss_fn(out.reshape(-1, vocab_size), y_batch.reshape(-1))

            test_loss += loss.item()
            test_batches += 1

            preds = out.argmax(dim=-1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.numel()

    test_loss = test_loss / test_batches
    test_acc = correct / total

    # =========================
    # scheduler step
    # =========================
    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        scheduler.step(test_loss)
    else:
        scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    torch.save(model.state_dict(), f"ckpt/model_epoch{epoch+1}.pth")

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Train Loss: {train_loss:.4f}, "
        f"Test Loss: {test_loss:.4f}, "
        f"Test Acc: {test_acc:.4f}, "
        f"LR: {current_lr:.6f}"
    )

Epoch 1/1000, Train Loss: 3.1158, Test Loss: 2.6643, Test Acc: 0.2865, LR: 0.000300
Epoch 2/1000, Train Loss: 2.6114, Test Loss: 2.4799, Test Acc: 0.3079, LR: 0.000300
Epoch 3/1000, Train Loss: 2.4749, Test Loss: 2.3625, Test Acc: 0.3284, LR: 0.000300
Epoch 4/1000, Train Loss: 2.3675, Test Loss: 2.2421, Test Acc: 0.3565, LR: 0.000300
Epoch 5/1000, Train Loss: 2.2748, Test Loss: 2.1462, Test Acc: 0.3801, LR: 0.000300
Epoch 6/1000, Train Loss: 2.2012, Test Loss: 2.0713, Test Acc: 0.3992, LR: 0.000300
Epoch 7/1000, Train Loss: 2.1353, Test Loss: 2.0028, Test Acc: 0.4137, LR: 0.000300
Epoch 8/1000, Train Loss: 2.0727, Test Loss: 1.9361, Test Acc: 0.4301, LR: 0.000300
Epoch 9/1000, Train Loss: 2.0131, Test Loss: 1.8765, Test Acc: 0.4463, LR: 0.000300
Epoch 10/1000, Train Loss: 1.9618, Test Loss: 1.8282, Test Acc: 0.4601, LR: 0.000300
Epoch 11/1000, Train Loss: 1.9186, Test Loss: 1.7882, Test Acc: 0.4713, LR: 0.000300
Epoch 12/1000, Train Loss: 1.8818, Test Loss: 1.7537, Test Acc: 0.4810, LR

KeyboardInterrupt: 